# Greyline and Terminator Analysis

Explore propagation at the day/night boundary (solar terminator).

**What you'll learn:**
- Classify paths by solar geometry (both-day, cross-terminator, both-dark)
- Understand greyline enhancement on low bands
- Analyze terminator effects on HF propagation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone

from ionis_jupyter import (
    load_dataset,
    grid_to_latlon,
    solar_elevation,
    is_dark,
    classify_path,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

In [ ]:
df = load_dataset("wspr")
print(f"Loaded {len(df):,} signatures")

## What is Greyline Propagation?

**The solar terminator** is the boundary between day and night on Earth. It moves at ~1,670 km/h at the equator.

**Greyline propagation** occurs when both ends of a path are near the terminator:
- D-layer absorption is reduced (twilight = less UV ionization)
- F-layer is still ionized from recent daylight
- Result: enhanced low-band (80m, 160m) DX for ~30-60 minutes

**Path classification:**
- **Both-day:** TX and RX both in daylight
- **Cross-terminator:** One in day, one in darkness
- **Both-dark:** TX and RX both in darkness

In [ ]:
# Demonstrate solar elevation calculation
grid = "DN13"  # Idaho
lat, lon = grid_to_latlon(grid)

print(f"Grid: {grid} ({lat:.2f}°N, {lon:.2f}°)")
print()

# Check elevation at different hours on March 1
print("Solar elevation on March 1, 2026:")
for hour in [0, 6, 12, 14, 18, 22]:
    dt = datetime(2026, 3, 1, hour, 0, tzinfo=timezone.utc)
    elev = solar_elevation(lat, lon, dt)
    status = "dark" if elev < -6 else "twilight" if elev < 0 else "day"
    print(f"  {hour:02d}:00 UTC: {elev:+.1f}° ({status})")

In [ ]:
# Classify a path
tx_grid = "DN13"  # Idaho
rx_grid = "JN48"  # Central Europe

print(f"Path: {tx_grid} → {rx_grid}")
print()

# Check classification at different hours
print("Path classification on March 1, 2026:")
for hour in range(0, 24, 2):
    dt = datetime(2026, 3, 1, hour, 0, tzinfo=timezone.utc)
    classification = classify_path(tx_grid, rx_grid, dt)
    
    tx_elev = solar_elevation(*grid_to_latlon(tx_grid), dt)
    rx_elev = solar_elevation(*grid_to_latlon(rx_grid), dt)
    
    print(f"  {hour:02d}:00 UTC: {classification:18} (TX: {tx_elev:+.0f}°, RX: {rx_elev:+.0f}°)")

In [ ]:
# Classify all paths in the dataset
# This is computationally intensive - using a sample

sample = df.sample(min(50000, len(df)), random_state=42).copy()

def classify_row(row):
    """Classify a single row by solar geometry."""
    # Use March 15 as representative date (equinox)
    dt = datetime(2026, 3, 15, int(row["hour"]), 0, tzinfo=timezone.utc)
    return classify_path(row["tx_grid_4"], row["rx_grid_4"], dt)

print("Classifying paths (this may take a minute)...")
sample["path_class"] = sample.apply(classify_row, axis=1)

# Distribution
print("\nPath classification distribution:")
print(sample["path_class"].value_counts())

In [ ]:
# SNR by path classification
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: All bands
class_snr = sample.groupby("path_class")["median_snr"].mean().sort_values()
axes[0].barh(class_snr.index, class_snr.values, color="steelblue", alpha=0.7)
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Mean SNR (dB)")
axes[0].set_title("SNR by Path Classification (All Bands)")

# Right: By band
BAND_NAMES = {103: "80m", 105: "40m", 107: "20m", 109: "15m", 111: "10m"}
pivot = sample.pivot_table(
    values="median_snr", 
    index="path_class", 
    columns="band", 
    aggfunc="mean"
)
pivot = pivot[[b for b in [103, 105, 107, 109, 111] if b in pivot.columns]]
pivot.columns = [BAND_NAMES.get(b, str(b)) for b in pivot.columns]

pivot.plot(kind="bar", ax=axes[1], alpha=0.7)
axes[1].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Path Classification")
axes[1].set_ylabel("Mean SNR (dB)")
axes[1].set_title("SNR by Classification and Band")
axes[1].legend(title="Band")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## Key Findings

**Low bands (80m, 160m):**
- Best when both ends are dark (D-layer gone)
- Cross-terminator can work but with fading
- Daytime paths usually dead (D-layer absorption)

**High bands (10m, 15m, 20m):**
- Best when both ends have daylight (F-layer ionized)
- Both-dark paths are dead (MUF too low)
- Cross-terminator can extend openings

**Greyline window:**
- ~30-60 minutes around local sunrise/sunset
- Look for cross-terminator paths where BOTH ends are in twilight